# Analytical Window Functions: Rankings, Lead/Lag, and Moving Averages

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 06_window_functions.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How to construct Window specifications, compute ranking functions (row_number, rank, dense_rank), analytical offsets (lead, lag), and cumulative running totals.
- **Why does this topic exist?**: Window functions solve advanced analytical questions (e.g. top N items per group, year-over-year changes, 7-day rolling averages) without self-joins.
- **What problem does it solve?**: Performing group-based ranking with self-joins causes severe performance degradation.
- **Where is it used in real-world Data Engineering?**: Financial reporting, fraud detection (time between transactions), and sales leaderboards.

---

## 2. Window Function Architecture

```
Window Specification:
Window.partitionBy("category").orderBy(desc("revenue")).rowsBetween(start, end)
  |                                 |                       |
  v Segregates data                 v Sorts inside frame    v Defines frame slice
[Electronics]                   [Highest -> Lowest]   [e.g. -3 to Current Row]
  |-- Row 1: Laptop ($1500)       -> row_number=1, rank=1, dense_rank=1
  |-- Row 2: Phone ($800)         -> row_number=2, rank=2, dense_rank=2
[Clothing]
  |-- Row 1: Jacket ($200)        -> row_number=1, rank=1, dense_rank=1
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Window_Functions_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Top 2 Highest-Grossing Orders per Channel

In [ ]:
from pyspark.sql.window import Window

orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")

# Define window specification
channel_window = Window.partitionBy("channel").orderBy(F.desc("quantity"))

ranked_orders = (
    orders
    .withColumn("row_num", F.row_number().over(channel_window))
    .withColumn("rank_num", F.rank().over(channel_window))
    .withColumn("dense_rank_num", F.dense_rank().over(channel_window))
    .filter(F.col("row_num") <= 2)
    .select("channel", "order_id", "customer_id", "quantity", "row_num", "rank_num", "dense_rank_num")
)

ranked_orders.show()

## 4. Analytical Offsets: `lag()` to Calculate Days Between Customer Orders

In [ ]:
customer_window = Window.partitionBy("customer_id").orderBy("order_ts")

order_intervals = (
    orders
    .withColumn("prev_order_ts", F.lag("order_ts", 1).over(customer_window))
    .withColumn("days_since_prev", F.datediff(F.col("order_ts"), F.col("prev_order_ts")))
    .filter(F.col("prev_order_ts").isNotNull())
    .select("customer_id", "order_id", "order_ts", "prev_order_ts", "days_since_prev")
)

order_intervals.show(5)

## 5. Running Totals and Moving Averages with `rowsBetween`

In [ ]:
# Cumulative revenue running sum per customer
running_window = Window.partitionBy("customer_id").orderBy("order_ts").rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_totals = (
    orders
    .withColumn("order_value", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("cumulative_spend", F.round(F.sum("order_value").over(running_window), 2))
    .select("customer_id", "order_ts", "order_value", "cumulative_spend")
)

running_totals.show(6)

---

## Experiment: rowsBetween vs rangeBetween Semantics

- **1. What are we testing?**: Demonstrate the difference between physical row offset (rowsBetween) and logical value range (rangeBetween).
- **2. What do we expect?**: rowsBetween counts physical row positions; rangeBetween groups tied order values.

In [ ]:
sample_data = spark.createDataFrame([
    ("A", 1, 100),
    ("A", 2, 100),
    ("A", 3, 200)
], ["user", "time_step", "value"])

w_rows = Window.partitionBy("user").orderBy("value").rowsBetween(Window.unboundedPreceding, Window.currentRow)
w_range = Window.partitionBy("user").orderBy("value").rangeBetween(Window.unboundedPreceding, Window.currentRow)

sample_data.withColumn("sum_rows", F.sum("value").over(w_rows)) \
           .withColumn("sum_range", F.sum("value").over(w_range)) \
           .show()

**3. What happened?**: For row 1 and 2 with tied value 100, sum_range immediately jumped to 200 for both rows, while sum_rows accumulated row by row.  
**4. Why did it happen?**: rangeBetween treats duplicate values in the orderBy column as peers within the same frame slice.  
**5. Practical Engineering Lesson**: Use rowsBetween when you need strict row-based moving averages (e.g. 3-row moving average).

---

## Practical Exercises

### Exercise 1 (Beginner): Top 1 Spending Customer per Region
**Problem Statement**: Join customers and orders, and find the single highest-spending customer in each region using row_number().  
**Input / Context**: customers and orders

In [ ]:
# Exercise 1: Top customer per region

### Exercise 2 (Intermediate): Percentage of Total Spend per Customer
**Problem Statement**: Calculate each individual order's percentage contribution to that customer's lifetime total spend.  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 2: Spend contribution

### Exercise 3 (Advanced): 3-Order Rolling Moving Average
**Problem Statement**: Compute a 3-order moving average of unit_price for each product over time (1 preceding, current, 1 following).  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 3: Rolling moving average

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Top 1 Spending Customer per Region):
```python
cust_spend = orders.join(customers,"customer_id").groupBy("region","customer_id").agg(F.sum(F.col("quantity")*F.col("unit_price")).alias("spend"))
w = Window.partitionBy("region").orderBy(F.desc("spend"))
top_custs = cust_spend.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
top_custs.show()
```

#### Solution 2 (Percentage of Total Spend per Customer):
```python
w_cust_all = Window.partitionBy("customer_id")
orders.withColumn("val", F.col("quantity")*F.col("unit_price")) \
      .withColumn("total_cust_spend", F.sum("val").over(w_cust_all)) \
      .withColumn("pct_of_total", F.round((F.col("val") / F.col("total_cust_spend")) * 100, 2)) \
      .select("customer_id","order_id","val","pct_of_total").show(5)
```

#### Solution 3 (3-Order Rolling Moving Average):
```python
w_roll = Window.partitionBy("product_id").orderBy("order_ts").rowsBetween(-1, 1)
orders.withColumn("moving_avg_price", F.round(F.avg("unit_price").over(w_roll), 2)) \
      .select("product_id","order_ts","unit_price","moving_avg_price").show(6)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Window Power**: Performs aggregations without collapsing rows into a single record.
- **Ranking Functions**: row_number() is unique; rank() skips numbers on ties; dense_rank() does not skip.
- **Lead and Lag**: Enables time-series difference calculations without self-joins.

### Essential Technical Terms
- `Window.partitionBy`: Defines the boundary of rows processed together.
- `rowsBetween`: Defines physical row offset frame.
- `rangeBetween`: Defines logical value range frame.

### Common Engineering Mistakes
- **Window without partitionBy on Large Data**: Forces all data in the entire dataset into a single partition on 1 executor, causing OOM.
- **Confusing rank() with dense_rank()**: Rank leaves gaps after ties; dense_rank is sequential.

### Interview Questions on This Topic
- **Q: What happens if you omit partitionBy() in a Window?**
  - *A*: Spark moves the entire dataset into a single partition on one executor, completely disabling parallelism and likely crashing with OOM.
- **Q: What is the difference between lag() and lead()?**
  - *A*: lag() looks at preceding rows; lead() looks at subsequent rows.

### What to Study Next
Open 03_dataframe_and_spark_sql/07_nulls_and_data_quality.ipynb to master dirty record handling.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')